In [35]:
import pandas as pd

from pyomo.environ import *

In [38]:
eco = pd.read_csv("Expected_Revenue_Economy_JFK.csv")
pre = pd.read_csv("Expected_Revenue_Premium_JFK.csv")
bus = pd.read_csv("Expected_Revenue_Business_JFK.csv")

eco["Fare"] = "Economy"
pre["Fare"] = "Premium"
bus["Fare"] = "Business"

df = pd.concat([eco, pre, bus], ignore_index=True)

df.head()

,Booking Stage,Price,Demand,Expected Revenue,Fare
0,Early,100,102.254161,10225.416149,Economy
1,Early,110,102.254161,11247.957764,Economy
2,Early,120,101.186528,12142.383319,Economy
3,Early,130,98.706367,12831.827679,Economy
4,Early,140,95.406283,13356.879656,Economy


In [39]:
capacity = {

    "Economy":165,

    "Premium":33,

    "Business":22

}

In [40]:
price = {}
demand = {}
revenue = {}

for _, row in df.iterrows():

    key = (
        row["Fare"],
        row["Booking Stage"],
        row["Price"]
    )

    price[key] = row["Price"]

    demand[key] = row["Demand"]

    revenue[key] = row["Expected Revenue"]

In [41]:
fares = sorted(df["Fare"].unique())

stages = ["Early","Middle","Final"]

prices = {}

for fare in fares:

    for stage in stages:

        prices[(fare,stage)] = sorted(

            df[
                (df["Fare"]==fare)
                &
                (df["Booking Stage"]==stage)
            ]["Price"].unique()

        )

In [42]:
model = ConcreteModel()

In [43]:
model.x = Var(

    [

        (f,s,p)

        for f in fares

        for s in stages

        for p in prices[(f,s)]

    ],

    within=Binary

)

In [44]:
def objective(model):

    return sum(

        revenue[(f,s,p)] *

        model.x[f,s,p]

        for f in fares

        for s in stages

        for p in prices[(f,s)]

    )

model.obj = Objective(

    rule=objective,

    sense=maximize

)

In [45]:
def one_price(model,f,s):

    return sum(

        model.x[f,s,p]

        for p in prices[(f,s)]

    ) == 1

model.one_price = Constraint(

    fares,

    stages,

    rule=one_price

)

In [46]:
def max_capacity(model,f):

    return sum(

        demand[(f,s,p)] *

        model.x[f,s,p]

        for s in stages

        for p in prices[(f,s)]

    ) <= capacity[f]

model.max_capacity = Constraint(

    fares,

    rule=max_capacity

)

In [47]:
MIN_LOAD_FACTOR = 0.95

def min_capacity(model,f):

    return sum(

        demand[(f,s,p)] *

        model.x[f,s,p]

        for s in stages

        for p in prices[(f,s)]

    ) >= MIN_LOAD_FACTOR * capacity[f]

model.min_capacity = Constraint(

    fares,

    rule=min_capacity

)

In [48]:
solver = SolverFactory("gurobi")

results = solver.solve(model)

print(results.solver.status)

print(results.solver.termination_condition)

ok
optimal


In [49]:
solution=[]

for f in fares:

    for s in stages:

        for p in prices[(f,s)]:

            if value(model.x[f,s,p]) > 0.5:

                solution.append({

                    "Fare":f,

                    "Stage":s,

                    "Price":p,

                    "Demand":round(demand[(f,s,p)],2),

                    "Revenue":round(revenue[(f,s,p)],2)

                })

solution = pd.DataFrame(solution)

solution

,Fare,Stage,Price,Demand,Revenue
0,Business,Early,970,2.28,2208.82
1,Business,Middle,990,7.48,7405.14
2,Business,Final,940,12.23,11498.82
3,Economy,Early,230,55.50,12764.54
4,Economy,Middle,250,70.68,17670.53
5,Economy,Final,250,38.69,9671.67
6,Premium,Early,530,6.87,3640.25
7,Premium,Middle,510,16.03,8176.52
8,Premium,Final,500,10.06,5027.66


In [50]:
summary = solution.groupby("Fare").agg({

    "Demand":"sum",

    "Revenue":"sum"

})

summary["Capacity"] = summary.index.map(capacity)

summary["Load Factor"] = (

    summary["Demand"]

    /

    summary["Capacity"]

).round(3)

summary

,Demand,Revenue,Capacity,Load Factor
Fare,,,,
Business,21.99,21112.78,22,1.000
Economy,164.87,40106.74,165,0.999
Premium,32.96,16844.43,33,0.999
